## 1. Title
# Benchmarking Deep Learning Architectures for Financial Time-Series Prediction and Risk Optimization under Market Regimes
This is the method and navigation notebook. Run `10_data_audit.ipynb`, then `20_benchmark.ipynb`, `30_validation.ipynb`, and `40_report.ipynb`. No model trains on opening this notebook.

# 20 — Controlled architecture benchmark
All outputs start empty. The **OPTIONAL — EXECUTE TO TRAIN** cell runs only when `RUN_TRAINING=1`. Set `FIN_DATA_PATH` and comma-separated `FIN_ASSETS` after auditing the CSV. `MODEL` is `all`, `dlinear`, `lstm`, `patchtst`, or `mamba`. Requesting Mamba explicitly requires genuine `mamba_ssm` and fails if it is missing.

## 2. Research Problem
Generate bounded, risk-aware directional signals for a multi-asset portfolio using historical windows only. Compare predictive accuracy with realized economic performance.

## 3. Research Gap
This independent benchmark tests external validity on US and Indian markets, regime-dependent rankings, predictive versus economic performance, and the return on architectural complexity. It extends, rather than claims to originate, standardized financial architecture benchmarking.

## 4. Research Questions
Which architecture has the strongest out-of-sample risk-adjusted performance? Does its rank survive high-volatility and bearish regimes? Do lower prediction errors improve net Sharpe? Is added complexity justified after costs?

## 5. Hypotheses
Temporal representation may improve Sharpe; the aggregate and crisis winners may differ; predictive and economic rankings may diverge; complexity may have diminishing economic returns. All are empirical hypotheses, not findings.

## 6. Dataset
Primary source: Global & Indian Financial Markets Dataset 2025. Inspect the actual local files and provenance in `10_data_audit.ipynb` before claims about columns, dates, or license. Secondary S&P 500 constituent data is optional and has potential survivorship bias.

## 7. Environment
Use the pinned CUDA container described in `RunningNotebooks.md` and `DOCKER.md`. The genuine Mamba model is required for benchmark output; a missing or incompatible `mamba_ssm` installation fails preflight rather than being relabelled as another architecture.

In [ ]:
from pathlib import Path
import os, random, json, re, sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
get_ipython().run_line_magic('run', str(ROOT / 'notebooks' / '00_core.ipynb'))
from src.data.adapters import GlobalIndianAdapter
from src.data.features import build_features
from src.data.windows import create_rolling_windows
from src.data.scaling import TimeSeriesScaler
from src.data.walk_forward import walk_forward_splits
from src.regimes.rules import detect_regimes
from src.backtest.backtester import Backtester
from src.portfolio.baselines import get_zero_signal_baseline, get_long_only_baseline, get_momentum_baseline, get_volatility_managed_baseline
from src.evaluation.metrics import compute_all_metrics
from src.evaluation.regime_metrics import evaluate_by_regime
from src.evaluation.analysis import predictive_metrics
from src.runtime.experiment import atomic_json_dump
RUN_TRAINING = os.getenv('RUN_TRAINING', '0') == '1'
MODEL = os.getenv('MODEL', 'dlinear').lower()
SEED = int(os.getenv('SEED', '42'))
FOLD = int(os.getenv('FOLD', '0'))
DEVICE = os.getenv('DEVICE', 'auto').lower()
DEVICE = 'cuda' if DEVICE == 'gpu' else DEVICE
RESUME_MODE = os.getenv('RESUME_MODE', 'auto').lower()
EPOCHS = int(os.getenv('EPOCHS', '0'))
RUN_ID = os.getenv('RUN_ID', '').strip()
OUTPUT_ROOT = Path(os.getenv('OUTPUT_ROOT', str(ROOT / 'outputs'))).resolve()
configs = load_project_configs(ROOT)
WALK_FORWARD_MODE = os.getenv('WALK_FORWARD_MODE', configs['backtest']['walk_forward'].get('mode', 'expanding')).lower()
assert MODEL in {'all', 'dlinear', 'lstm', 'mamba', 'patchtst'}
assert DEVICE in {'auto', 'cpu', 'cuda'}
assert RESUME_MODE in {'auto', 'restart', 'fail'}
assert WALK_FORWARD_MODE in {'expanding', 'rolling'}
assert FOLD >= 0 and EPOCHS >= 0
assert OUTPUT_ROOT.is_relative_to(ROOT), 'OUTPUT_ROOT must stay inside the repository'
if RUN_TRAINING:
    if not RUN_ID or not re.fullmatch(r'[A-Za-z0-9._-]+', RUN_ID): raise ValueError('RUN_ID must be a safe nonempty folder name')
    if DEVICE == 'cuda' and not torch.cuda.is_available(): raise RuntimeError('DEVICE=gpu/cuda requested but CUDA is unavailable')
print({'training': RUN_TRAINING, 'model': MODEL, 'seed': SEED, 'fold': FOLD, 'device': DEVICE, 'run_id': RUN_ID})

## 8. Reproducibility
Record input hashes, config files, Python and dependency versions, seed, machine, device, and Git revision. `RUN_TRAINING` defaults to false. Keep generated results separate from source notebooks.

## 9. Data Loading
Load the audited local CSV through `src.data.adapters.GlobalIndianAdapter`; set `FIN_DATA_PATH` and optional `FIN_MACRO_PATH`. Do not silently substitute a network series.

In [ ]:
def prepare_data():
    default_data = ROOT / 'data' / 'kaggle_global_indian_markets' / 'global-india-markets-macro' / 'daily_market_data.csv'
    data_path = Path(os.getenv('FIN_DATA_PATH', str(default_data))).resolve()
    assert data_path.is_relative_to(ROOT), 'Dataset must be inside the repository'
    assets = [x.strip() for x in os.getenv('FIN_ASSETS', '').split(',') if x.strip()]
    if not assets: raise ValueError('Set FIN_ASSETS to audited price columns before training')
    prices_all = GlobalIndianAdapter(data_path=str(data_path)).load()
    missing = sorted(set(assets) - set(prices_all.columns))
    if missing: raise ValueError(f'Unknown asset columns: {missing}')
    prices = prices_all.loc[:, assets].dropna()
    assert prices.index.is_monotonic_increasing and prices.index.is_unique
    assert (prices > 0).all().all(), 'Selected fields must be positive prices'
    features, returns, vol, _ = build_features(prices)
    wf = configs['backtest']['walk_forward']
    folds = list(walk_forward_splits(len(features), int(wf['initial_train_size']),
                                    int(wf['test_size']), step_size=int(wf['step_size']),
                                    mode=WALK_FORWARD_MODE, train_window=wf.get('rolling_train_window'),
                                    gap=int(wf.get('gap', 0))))
    if FOLD >= len(folds): raise ValueError(f'FOLD={FOLD} outside available 0..{len(folds)-1}')
    fold_train, fold_test = folds[FOLD]
    lookback = int(configs['data'].get('lookback', 20))
    val_size = max(lookback + 2, int(len(fold_train) * .2))
    if len(fold_train) <= val_size + lookback: raise ValueError('Insufficient train history for validation and lookback')
    fit_idx, val_idx = fold_train[:-val_size], fold_train[-val_size:]
    scaler = TimeSeriesScaler(); scaler.fit(features.iloc[fit_idx].to_numpy())
    scaled = scaler.transform(features.to_numpy())
    X, y, decision_dates = create_rolling_windows(scaled, returns.to_numpy(), lookback, features.index)
    target_dates = features.index[lookback:]
    assert len(X) == len(y) == len(decision_dates) == len(target_dates)
    assert (target_dates > decision_dates).all()
    boundaries = {'train': features.index[fit_idx], 'val': features.index[val_idx], 'test': features.index[fold_test]}
    masks = {name: np.asarray(decision_dates.isin(idx) & target_dates.isin(idx)) for name, idx in boundaries.items()}
    assert all(mask.sum() >= 2 for mask in masks.values())
    assert boundaries['train'].max() < boundaries['val'].min() < boundaries['test'].min()
    regimes = detect_regimes(prices).reindex(decision_dates)
    ex_ante_vol = vol.reindex(decision_dates).to_numpy()
    return {'prices': prices, 'returns_full': returns, 'X': X, 'y': y, 'dates': decision_dates,
            'target_dates': target_dates, 'vol': ex_ante_vol, 'regimes': regimes,
            'masks': masks, 'assets': assets, 'data_path': data_path,
            'fold_bounds': {k: [str(v.min()), str(v.max())] for k, v in boundaries.items()}}

## 10. Data Quality Checks
The audit notebook reports files, schema, date coverage, duplicates, missing values, and candidate asset columns. Review price-like versus rate-like fields before using percentage returns.

## 11. Exploratory Analysis
Inspect normalized prices, return distributions, rolling volatility, missingness, and correlation. These are descriptive views and must not set test-time thresholds.

## 12. Feature Engineering
`src.data.features.build_features` uses trailing returns, volatility, and momentum. Its aligned feature, return, and volatility outputs feed every architecture identically.

## 13. Leakage Checks
Assert increasing unique timestamps, feature/target alignment, training-only scaler fit, and no use of test metrics for tuning. The target is the return after the final input timestamp.

## 14. Temporal Data Splitting
Use chronological 60/20/20 train/validation/test partitions. Walk-forward folds can be built with `expanding_window_split` or `rolling_window_split` after data coverage is known.

## 15. Rolling Window Construction
`create_rolling_windows` yields `[sample, lookback, features]`, next-period returns, and decision dates. The final feature row has no observed next-period target and is excluded.

## 16. Regime Construction
`detect_regimes` uses trailing volatility and trend at the decision time. Regime thresholds and the warm-up convention must be documented; never use future returns for a label.

## 17. Portfolio Construction
Convert tanh signals to volatility-scaled positions with a configurable 10% target and 3x cap. Equal-weight eligible asset strategy returns; track cap frequency and unavailable assets.

## 18. Sharpe Objective
`NegativeSharpeLoss` pools portfolio returns across chronological sequence batches. `loss_alpha=1` is the default pure Sharpe objective; any hybrid predictive loss is a sensitivity case.

## 19. DLinear Baseline
`src.models.dlinear.DLinear` decomposes the sequence into trend and residual, projects each over time, then outputs bounded asset signals. Document its nonlinear bottleneck as a DLinear-inspired adaptation.

## 20. LSTM
`src.models.lstm.LSTMSignalModel` uses the terminal hidden representation and tanh signal head.

## 21. Mamba
`src.models.mamba_model.MambaSignalModel` requires genuine `mamba_ssm`. If unavailable, fail the requested run and report the dependency error; do not relabel another model.

## 22. PatchTST
`src.models.patchtst.PatchTSTSignalModel` embeds temporal patches, applies Transformer encoding, and emits bounded asset signals.

## 23. Common Training Framework
`BaseTrainer` shares optimizer, Sharpe criterion, early stopping, and seed policy across architectures. Fit and tune on train/validation only. The optional training cell is in `20_benchmark.ipynb`.

### Shared models and training objective
Every model receives identical input and outputs one bounded signal per asset. Sequence batches are chronological and contain multiple observations so the pooled Sharpe objective is meaningful. The existing loss computes signal-times-return contributions; the actual volatility scaling is applied in the backtester and should be recognized when interpreting training versus economic objectives.

In [ ]:
def make_model(name, lookback, features, assets):
    return build_signal_model(name, lookback, features, assets, configs['models'])
def loader(X, y, batch_size):
    ds = TensorDataset(torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)

## 24. Backtesting Framework
`Backtester` sizes at decision time t and pairs positions with returns at t+1; transaction costs depend on position turnover. Sweep 0, 1, 5, 10, and 20 bps after training.

## 25. Risk Metrics
Use `compute_all_metrics` for return, volatility, Sharpe, Sortino, drawdown, Calmar, win rate, profit factor, and turnover. Review undefined-metric conventions before interpretation.

## 26. Regime Metrics
Use `evaluate_by_regime` on aligned out-of-sample returns and decision-time labels. Present mean, dispersion, worst-regime Sharpe, and best-minus-worst degradation separately.

## 27. Statistical Testing
Use block-bootstrap Sharpe intervals and paired comparisons on matched out-of-sample timestamps; disclose the hypotheses and number of comparisons.

## 28. Complexity Analysis
Track trainable parameters, training time, device, turnover, and net Sharpe per model and seed. Do not infer efficiency from parameter count alone.

## 29. Training Instructions
### OPTIONAL — EXECUTE TO TRAIN
Set `RUN_TRAINING=1` explicitly before running `20_benchmark.ipynb`. Review the audited asset mapping and split boundaries first. Full training and tuning are intentionally absent from the default path.

### OPTIONAL — EXECUTE TO TRAIN
Set `RUN_TRAINING=1` only after reviewing the audit and asset selection. The test set is used once per fitted seed for final out-of-sample evaluation, never for model selection. The run writes actual metrics into `results/notebook_run/` and records which architectures were unavailable.

In [ ]:
def evaluate_signals(name, signals, data, run_dir, seconds=None, parameters=None, history=None):
    test = data['masks']['test']
    dates = data['dates'][test]
    targets = data['y'][test]
    vol = data['vol'][test]
    fees = configs['backtest']['transaction_costs_bps']
    bt = Backtester(target_vol=float(configs['backtest']['target_volatility']),
                    max_leverage=float(configs['backtest']['max_leverage']), tc_bps=0)
    outcome = bt.run(signals, vol, targets, dates=dates)
    predictive = predictive_metrics(signals, targets)
    turnover = outcome['turnover']
    metrics_rows, regime_rows = [], []
    for bps in fees:
        net = outcome['gross_returns'] - turnover * (float(bps) / 10000.0)
        metrics = compute_all_metrics(net, turnover)
        metrics_rows.append({'model':name,'seed':SEED,'fold':FOLD,'cost_bps':float(bps),
                             'train_seconds':seconds,'parameters':parameters,
                             'cap_fraction':float(np.mean(np.abs(outcome['positions'].to_numpy()) >= float(configs['backtest']['max_leverage']) - 1e-9)),
                             **predictive,**metrics})
        for regime, row in evaluate_by_regime(net, data['regimes'][test], confidence_intervals=True, block_size=int(configs['backtest']['bootstrap']['block_size']), num_bootstraps=int(configs['backtest']['bootstrap']['num_bootstraps'])).iterrows():
            regime_rows.append({'model':name,'seed':SEED,'fold':FOLD,'cost_bps':float(bps),'regime':regime,**row.to_dict()})
    run_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(metrics_rows).to_csv(run_dir / 'metrics.csv', index=False)
    pd.DataFrame(regime_rows).to_csv(run_dir / 'regimes.csv', index=False)
    pd.DataFrame({'decision_date':dates.astype(str),'target_date':data['target_dates'][test].astype(str),
                  'gross_return':outcome['gross_returns'].to_numpy(),'turnover':turnover.to_numpy(),
                  'model':name,'seed':SEED,'fold':FOLD}).to_csv(run_dir / 'returns.csv', index=False)
    pd.DataFrame({'decision_date':np.repeat(dates.astype(str),len(data['assets'])),
                  'target_date':np.repeat(data['target_dates'][test].astype(str),len(data['assets'])),
                  'asset':np.tile(data['assets'],len(dates)),'signal':signals.reshape(-1),'position':outcome['positions'].to_numpy().reshape(-1),
                  'target_return':targets.reshape(-1),'model':name,'seed':SEED,'fold':FOLD}).to_csv(run_dir / 'predictions.csv', index=False)
    return metrics_rows
if RUN_TRAINING:
    data = prepare_data()
    names = ['dlinear','lstm','mamba','patchtst'] if MODEL == 'all' else [MODEL]
    # Preflight all requested architectures; genuine Mamba is mandatory when requested.
    for name in names: make_model(name,data['X'].shape[1],data['X'].shape[2],len(data['assets']))
    tr = dict(configs['training']); batch_size = int(tr['batch_size'])
    if batch_size < 2: raise ValueError('Sharpe batches require at least two decisions')
    train, val, test = (data['masks'][key] for key in ('train','val','test'))
    train_loader = loader(data['X'][train],data['y'][train],batch_size)
    val_loader = loader(data['X'][val],data['y'][val],batch_size)
    test_loader = loader(data['X'][test],data['y'][test],batch_size)
    for name in names:
        unit = NotebookUnitRun(OUTPUT_ROOT,RUN_ID,name,SEED,FOLD,RESUME_MODE)
        manifest_path = OUTPUT_ROOT / RUN_ID / 'metadata' / 'data_manifest.json'
        lock_path = OUTPUT_ROOT / RUN_ID / 'metadata' / 'dependency_lock.txt'
        unit.checkpoint_metadata = {
            'git_revision': environment_record(ROOT,data['data_path']).get('git_revision'),
            'data_manifest_hash': sha256_file(manifest_path) if manifest_path.exists() else None,
            'resolved_config_hash': hashlib.sha256(json.dumps(configs,sort_keys=True).encode()).hexdigest(),
            'dependency_lock_hash': sha256_file(lock_path) if lock_path.exists() else None,
            'image_identifier': os.getenv('BENCHMARK_IMAGE'), **unit.identity}
        unit.event('unit_start',model=name,seed=SEED,fold=FOLD,device=DEVICE,epochs=EPOCHS)
        # Bounded validation-only tuning. The test loader is never used to rank candidates.
        import copy
        search = tr.get('hyperparameter_search', {})
        candidates = list(search.get('learning_rates', [tr['learning_rate']]))[:int(search.get('max_trials', 1))]
        tuning_epochs = min(int(search.get('tuning_epochs', 3)), EPOCHS or int(tr['epochs']))
        trials = []
        for learning_rate in candidates:
            trial_configs = copy.deepcopy(configs); trial_configs['training']['learning_rate'] = float(learning_rate)
            _, _, _, trial_seconds, validation_signals = train_and_predict(
                name,SEED,train_loader,val_loader,val_loader,
                (data['X'].shape[1],data['X'].shape[2],len(data['assets'])),trial_configs,
                run=None,resume=False,device=DEVICE,epochs=tuning_epochs)
            score = predictive_metrics(validation_signals,data['y'][val])['mse']
            trials.append({'learning_rate':float(learning_rate),'validation_mse':float(score),'seconds':trial_seconds})
        selected = min(trials,key=lambda row:row['validation_mse'])
        selected_configs = copy.deepcopy(configs); selected_configs['training']['learning_rate'] = selected['learning_rate']
        atomic_json_dump({'selection_metric':'validation_mse','test_used_for_selection':False,'trials':trials,'selected':selected}, unit.table_dir / 'hyperparameter_search.json')
        model, optimizer, history, seconds, signals = train_and_predict(
            name,SEED,train_loader,val_loader,test_loader,
            (data['X'].shape[1],data['X'].shape[2],len(data['assets'])),selected_configs,
            run=unit,resume=RESUME_MODE == 'auto',device=DEVICE,
            epochs=EPOCHS if EPOCHS else None)
        rows = evaluate_signals(name,signals,data,unit.table_dir,seconds,model.count_parameters(),history)
        atomic_json_dump({'model':name,'seed':SEED,'fold':FOLD,'assets':data['assets'],
                          'fold_bounds':data['fold_bounds'],'environment':environment_record(ROOT,data['data_path']),
                          'config':selected_configs,'hyperparameter_selection':selected,'predictive_metrics':predictive_metrics(signals,data['y'][test])},
                         unit.table_dir / 'metadata.json')
        unit.event('unit_complete',metrics=rows,seconds=seconds)
        fig,_ = plot_training_history(history,title=f'{name} seed {SEED} fold {FOLD}')
        figure_dir = OUTPUT_ROOT / RUN_ID / 'figures' / name / f'seed_{SEED}'
        figure_dir.mkdir(parents=True,exist_ok=True)
        fig.savefig(figure_dir / f'fold_{FOLD}_loss.png',dpi=150,bbox_inches='tight')
        import matplotlib.pyplot as plt
        plt.close(fig)
    # Four transparent baselines use the same test dates, portfolio sizing and cost sweep.
    test_dates = data['dates'][test]
    all_returns = data['returns_full'].to_numpy()
    decision_positions = data['returns_full'].index.get_indexer(test_dates)
    lookback = int(configs['backtest']['baselines']['momentum_lookback'])
    vol_lookback = int(configs['backtest']['baselines']['volatility_lookback'])
    equal_weight_signal = np.clip(data['vol'][test] / np.nanmax(data['vol'][test],axis=1,keepdims=True),0,1)
    baseline_signals = {
        'baseline_equal_weight': equal_weight_signal,
        'baseline_zero': get_zero_signal_baseline(data['y'][test]),
        'baseline_long_only': get_long_only_baseline(data['y'][test]),
        'baseline_momentum': get_momentum_baseline(all_returns,lookback)[decision_positions],
        'baseline_volatility_managed': np.clip(get_volatility_managed_baseline(
            all_returns,float(configs['backtest']['target_volatility']),vol_lookback)[decision_positions],-1,1)}
    for name, signals in baseline_signals.items():
        target = OUTPUT_ROOT / RUN_ID / 'tables' / name / f'seed_{SEED}' / f'fold_{FOLD}'
        if not (target / 'metrics.csv').exists():
            evaluate_signals(name,signals,data,target)
else:
    print('Training disabled. Set RUN_TRAINING=1 with MODEL, SEED, FOLD and RUN_ID to execute one unit.')

### Interpretation guardrails
Every requested Mamba run, including MODEL=all, fails if the genuine mamba_ssm implementation is unavailable. Incomplete runs are excluded from ranking. These notebook results are new executions, not copies of archived runs. Inspect train loss curves, cap frequency, missing assets, and timestamp alignment before scientific claims.

## 30. Results Template
`40_report.ipynb` builds empty tables until actual benchmark artifacts exist. Required columns include model, seed, split, gross/net Sharpe, regime, turnover, cost, complexity, and provenance.

## 31. Expected Analysis Questions
Does the leading model remain strongest in stress? Does turnover erase gross advantage? How sensitive is rank to seed and fees? Are Sharpe differences statistically distinguishable?

## 32. Limitations
Dataset quality and license require verification. Current-constituent robustness data may suffer survivorship bias. Rule-based regimes are threshold-sensitive. Backtests exclude live execution effects.

## 33. Reproducibility Checklist
Before reporting: audit data and license; verify temporal alignment and training-only transforms; record configs and hashes; run multiple seeds; preserve untouched test; report Mamba availability, costs, uncertainty, and all exclusions. Never fabricate metrics.